In [1]:
import os, cv2, gc, math, timm, torch
import numpy as np, pandas as pd
from tqdm import tqdm
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
import albumentations as A
from albumentations.pytorch import ToTensorV2

DEVICE = "cuda"


/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [2]:
IMG_DIR = "/kaggle/input/rsna-bone-age/boneage-training-dataset/boneage-training-dataset"
CSV_PATH = "/kaggle/input/rsna-bone-age/boneage-training-dataset.csv"
ROI_DIR = "/kaggle/working/roi_images"
os.makedirs(ROI_DIR, exist_ok=True)


In [3]:
df = pd.read_csv(CSV_PATH)
df["male"] = df["male"].astype(int)


In [4]:
class AttDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
        self.tf = A.Compose([
            A.Resize(224,224),
            A.Normalize(),
            ToTensorV2()
        ])
        
    def __len__(self): return len(self.df)
    
    def __getitem__(self, i):
        r = self.df.iloc[i]
        img = cv2.imread(f"{IMG_DIR}/{r.id}.png", 0)
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        img = self.tf(image=img)["image"]
        return img, r.boneage


In [5]:
class AttentionNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model("resnet18", pretrained=True, num_classes=0)
        self.fc = nn.Linear(512, 240)
        
    def forward(self, x):
        f = self.backbone(x)
        return self.fc(f)


In [6]:
train_df, val_df = train_test_split(df, test_size=0.1, random_state=42)

train_loader = DataLoader(AttDataset(train_df), batch_size=64, shuffle=True, num_workers=2)
val_loader = DataLoader(AttDataset(val_df), batch_size=64)

model_att = AttentionNet().to(DEVICE)
opt = torch.optim.Adam(model_att.parameters(), 1e-3)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(10):
    model_att.train()
    for x,y in tqdm(train_loader):
        x,y = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad()
        loss = loss_fn(model_att(x), y)
        loss.backward()
        opt.step()
    print("epoch",epoch,"done")


model.safetensors:   0%|          | 0.00/46.8M [00:00<?, ?B/s]

100%|██████████| 178/178 [03:58<00:00,  1.34s/it]


epoch 0 done


100%|██████████| 178/178 [02:53<00:00,  1.03it/s]


epoch 1 done


...

epoch 7 done


100%|██████████| 178/178 [03:13<00:00,  1.09s/it]


epoch 8 done


100%|██████████| 178/178 [02:49<00:00,  1.05it/s]

epoch 9 done


In [7]:
def extract_roi(img, cam):
    cam = cv2.resize(cam, img.shape[::-1])
    cam = (cam-cam.min())/(cam.max()+1e-8)
    mask = cam > 0.4
    ys,xs = np.where(mask)
    if len(xs)==0:
        return cv2.resize(img,(384,384))
    y1,y2 = ys.min(), ys.max()
    x1,x2 = xs.min(), xs.max()
    pad = 15
    y1,y2 = max(0,y1-pad), min(img.shape[0],y2+pad)
    x1,x2 = max(0,x1-pad), min(img.shape[1],x2+pad)
    crop = img[y1:y2,x1:x2]
    return cv2.resize(crop,(384,384))


In [8]:
model_att.eval()

for _,r in tqdm(df.iterrows(), total=len(df)):
    img = cv2.imread(f"{IMG_DIR}/{r.id}.png",0)
    inp = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    inp = A.Resize(224,224)(image=inp)["image"]
    inp = torch.from_numpy(inp).permute(2,0,1).float().unsqueeze(0)/255
    inp = inp.to(DEVICE)
    
    with torch.no_grad():
        feat = model_att.backbone(inp)
        logits = model_att.fc(feat)
        cls = logits.argmax(1).item()
        w = model_att.fc.weight[cls]
        fmap = model_att.backbone.forward_features(inp)
        cam = (fmap[0]*w[:,None,None]).sum(0).cpu().numpy()
    
    roi = extract_roi(img, cam)
    cv2.imwrite(f"{ROI_DIR}/{r.id}.png", roi)


100%|██████████| 12611/12611 [13:53<00:00, 15.13it/s]


In [9]:
def make_dist(age, sigma=2.5):
    x = np.arange(240)
    p = np.exp(-(x-age)**2/(2*sigma*sigma))
    return p/p.sum()


In [10]:
class BoneDataset(Dataset):
    def __init__(self, df, train=True):
        self.df = df.reset_index(drop=True)
        self.tf = A.Compose([
            A.HorizontalFlip(p=0.5),
            A.Rotate(10,p=0.5),
            A.Normalize(),
            ToTensorV2()
        ]) if train else A.Compose([
            A.Normalize(),
            ToTensorV2()
        ])
        
    def __len__(self): return len(self.df)
    
    def __getitem__(self,i):
        r = self.df.iloc[i]
        img = cv2.imread(f"{ROI_DIR}/{r.id}.png")
        img = self.tf(image=img)["image"]
        dist = make_dist(r.boneage)
        return img, torch.tensor(dist).float(), torch.tensor(r.boneage).float(), torch.tensor(r.male)


In [11]:
class BoneNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model("convnext_tiny", pretrained=True, num_classes=0)
        self.gender = nn.Embedding(2,32)
        self.dist = nn.Linear(768+32,240)
        self.reg = nn.Linear(768+32,1)
        
    def forward(self,x,g):
        f = self.backbone(x)
        g = self.gender(g)
        h = torch.cat([f,g],1)
        return F.log_softmax(self.dist(h),1), self.reg(h).squeeze(1)


In [12]:
train_df, val_df = train_test_split(df, test_size=0.1, random_state=42)

train_loader = DataLoader(BoneDataset(train_df,True), batch_size=24, shuffle=True, num_workers=2)
val_loader = DataLoader(BoneDataset(val_df,False), batch_size=48)


In [13]:
model = BoneNet().to(DEVICE)
opt = torch.optim.AdamW(model.parameters(),3e-4)


model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

In [14]:
def loss_fn(logp, dist, pred, age):
    kl = F.kl_div(logp, dist, reduction="batchmean")
    l1 = F.l1_loss(pred, age)
    return kl + 0.05*l1


In [15]:
best = 1e9

for epoch in range(25):
    model.train()
    for x,d,a,g in tqdm(train_loader):
        x,d,a,g = x.to(DEVICE),d.to(DEVICE),a.to(DEVICE),g.to(DEVICE)
        opt.zero_grad()
        lp,p = model(x,g)
        loss = loss_fn(lp,d,p,a)
        loss.backward()
        opt.step()
    
    model.eval()
    maes=[]
    with torch.no_grad():
        for x,d,a,g in val_loader:
            x,a,g = x.to(DEVICE),a.to(DEVICE),g.to(DEVICE)
            _,p = model(x,g)
            maes.append((p-a).abs().mean().item())
    mae = np.mean(maes)
    print("Epoch",epoch,"MAE",mae)
    if mae<best:
        best=mae
        torch.save(model.state_dict(),"best.pth")


100%|██████████| 473/473 [13:46<00:00,  1.75s/it]


Epoch 0 MAE 33.1782437077275


100%|██████████| 473/473 [13:41<00:00,  1.74s/it]


Epoch 1 MAE 31.552125718858505


100%|██████████| 473/473 [13:41<00:00,  1.74s/it]


...

Epoch 22 MAE 10.398146699976039


100%|██████████| 473/473 [13:41<00:00,  1.74s/it]


Epoch 23 MAE 10.444714899416324


100%|██████████| 473/473 [13:41<00:00,  1.74s/it]


Epoch 24 MAE 9.556808683607313
